# Fraud Detection with Multiple Classifiers

This notebook uses a simple 100-row fraud sample to explain preprocessing, EDA, model training, model comparison, and the MLOps lifecycle.

## 1. Import Libraries

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

## 2. Load and Prepare Data

In [ ]:
DATA_FILE = "creditcard.csv"
TARGET_COLUMN = "Class"
SAMPLE_SIZE = 100
RANDOM_STATE = 42

data = pd.read_csv(DATA_FILE)

fraud = data[data[TARGET_COLUMN] == 1].sample(SAMPLE_SIZE // 2, random_state=RANDOM_STATE)
non_fraud = data[data[TARGET_COLUMN] == 0].sample(SAMPLE_SIZE // 2, random_state=RANDOM_STATE)

data = pd.concat([fraud, non_fraud], ignore_index=True)
data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
data.columns = [column.lower() for column in data.columns]
data = data.drop_duplicates().reset_index(drop=True)

print("\nData shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head().to_string(index=False))
print("\nClass count:")
print(data["class"].value_counts().to_string())
print("\nMissing values:")
print(data.isnull().sum().to_string())

## 3. EDA Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

data["class"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["steelblue", "tomato"])
axes[0].set_title("Fraud vs Non-Fraud")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")

data.boxplot(column="amount", by="class", ax=axes[1])
axes[1].set_title("Amount by Class")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Amount")

data["amount"].plot(kind="hist", bins=15, ax=axes[2], color="seagreen", edgecolor="black")
axes[2].set_title("Amount Distribution")
axes[2].set_xlabel("Amount")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. Train Multiple Classification Models

In [ ]:
X = data.drop("class", axis=1)
y = data["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "SVC": SVC(probability=True, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

stacking_model = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ("svc", SVC(probability=True, random_state=RANDOM_STATE)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
models["Stacking Classifier"] = stacking_model

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model

    predictions = model.predict(X_test_scaled)
    probabilities = model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    results.append(
        {
            "Model": name,
            "Accuracy": round(accuracy, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1 Score": round(f1, 4),
            "ROC AUC": round(roc_auc, 4),
        }
    )

    print(f"\n{name}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")

results = pd.DataFrame(results).sort_values(by=["F1 Score", "ROC AUC"], ascending=False).reset_index(drop=True)
best_model_name = results.iloc[0]["Model"]
best_model_object = trained_models[best_model_name]
results

## 5. MLOps: Model Packaging

This is the first real MLOps step after training.

What we add here:
- save the best model
- save the scaler used during preprocessing
- save metadata like metrics, features, and training date

Why this matters:
- without packaging, the model only exists inside the notebook session
- with packaging, the model becomes a reusable artifact for deployment

In [ ]:
packaged_model_dir = Path("packaged_model")
packaged_model_dir.mkdir(exist_ok=True)

model_path = packaged_model_dir / "best_model.joblib"
scaler_path = packaged_model_dir / "scaler.joblib"
metadata_path = packaged_model_dir / "model_metadata.json"

joblib.dump(best_model_object, model_path)
joblib.dump(scaler, scaler_path)

best_model_metrics = results.iloc[0].to_dict()

metadata = {
    "model_name": best_model_name,
    "saved_at": datetime.now().isoformat(),
    "sample_size": SAMPLE_SIZE,
    "target_column": "class",
    "feature_columns": X.columns.tolist(),
    "metrics": best_model_metrics,
    "model_file": str(model_path),
    "scaler_file": str(scaler_path)
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Best model packaged successfully.")
print("Saved files:")
print(model_path)
print(scaler_path)
print(metadata_path)
print("\nSaved metadata preview:")
print(json.dumps(metadata, indent=2)[:1200])